# Lesson 07 - Паттерн планирања

Ова свеска демонстрира **Паттерн планирања** за AI агенте користећи Microsoft Agent Framework.
Научићете како да поделите комплексне захтеве за путовање на структурисане подзадатке, доделите их специјализованим агенатима,
и извршите резултујући план — све користећи структурисани излаз омогућен Pydantic моделима.


## Подешавање


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Расцепљење задатка

Расцепљење задатка је основа шаблона за планирање. Уместо да једном агенту доделимо задатак да самостално реши компликован захтев од почетка до краја, ми раздвајамо проблем на мање, јасно дефинисане **подзадатке**. Сваком подзадатку додељујемо специјалистичког агента (нпр. летови, хотели, активности) са јасним приоритетима и зависностима у редоследу.

Овај приступ пружа неколико предности:
- **Јасноћа**: сваки подзадатак има једну одговорност.
- **Паралелизам**: независни подзадатци могу да се извршавају истовремено.
- **Поузданост**: грешке су ограничене на појединачне подзадатке.
- **Праћење буџета**: трошкови се процењују по подзадатку и акумулирају.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Креирање агента за планирање са структурираним излазом

Агент за планирање делује као **координатор на рецепцији**. На основу захтева за путовање високог нивоа, производи структурирани `TravelPlan` — раздвајајући захтев на подзадатке, постављајући приоритете и идентификујући зависности тако да консијерж или извршни слој могу да обаве посао.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Извршавање плана помоћу специјализованих алата

Када агент на рецепцији направи структуриран план, **агент консијерж** га извршава.
Сваки специјализовани алат обрађује једну категорију подсмукова (летови, хотели, активности). Консијерж
понавља кроз подсмукове плана по редоследу зависности и прослеђује сваки одговарајућем алату.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Резиме

У овој лекцији сте научили **шаблон планирања** за AI агенте:

1. **Декомпозиција задатка** — агент на рецепцији планира сложени захтев за путовање у
   структуиране подзадатке користећи Pydantic моделе, додељујући сваки специјалистичком агенту са приоритетима
   и зависностима.
2. **Структурирани излаз** — Прослеђивањем `response_format` агент враћа валидирани
   објекат `TravelPlan` уместо слободног текста, чинећи накнадну обраду поузданом.
3. **Извршење плана** — агент консијерж пролази кроз подзадатке користећи специјалистичке алате
   (`book_flight`, `reserve_hotel`, `book_activity`) да спроведе план и пријави резултате.

Овај шаблон раздваја *шта радити* (планирање) од *како то урадити* (извршење), чинећи агенте
модуларнијим, лакшим за тестирање и проширење.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем АИ сервиса за превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, молимо вас да имате у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом матичном језику треба сматрати ауторитетним извором. За критичне информације препорука је коришћење професионалног људског превода. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
